# Experimental Schema-Evolution Pipeline

This notebook is the **Jupyter Notebook equivalent** of `run_experimental_pipeline.py`.

It preserves the original pipeline logic while restructuring the script into executable, documented notebook cells.

## Pipeline

The notebook orchestrates the existing components in the following order:

**U-Schema source elements → Retriever → CandidateResult → SemanticDecisionService → SemanticDecision → MDEValidator → MDEValidationResult → DecisionPolicy → DecisionResult → EvolutionPlanner → SchemaChange → MigrationBuilder → SQL**

The original implementation deliberately mocks only the retrieval and LLM boundaries using the controlled benchmark. The downstream domain/service components remain the real project implementations.

This makes the execution deterministic and avoids requiring a live vector store, embedding model, or LLM API key.

> **Important:** this is an integration-evidence run using a small controlled benchmark, not a scientific retrieval experiment.


# 1. Imports and Repository Configuration

This section imports the standard Python libraries and the existing project classes.

The repository root is determined relative to the notebook's expected location. The project root is then added to `sys.path` so that imports such as `src.domain...` work correctly.

The service classes imported here are **not reimplemented** in the notebook; they are reused from the project.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

# Adjust this if the notebook is stored somewhere other than the project root.
REPO_ROOT = Path.cwd()

if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

from src.domain.entities.experimental import (
    DecisionType,
    MDEStatus,
    SemanticDecision,
)
from src.domain.entities.rag_schema import (
    FieldType,
    KnowledgeBaseDocument,
    SourceField,
)
from src.domain.entities.schema import DataType, USchemaAttribute
from src.domain.services.decision_policy import (
    DecisionPolicy,
    DecisionThresholds,
)
from src.domain.services.evolution_planner import EvolutionPlanner
from src.domain.services.mde_validator import (
    MDEValidator,
    StructuralContext,
)
from src.domain.services.migration_builder import MigrationBuilder
from src.domain.services.relevance_policy import RelevancePolicy
from src.infrastructure.rag.scoring_system import HybridScoringSystem
from src.infrastructure.rag.semantic_decision_service import (
    SemanticDecisionService,
)

DEFAULT_BENCHMARK = (
    REPO_ROOT / "data/experimental/open_world/benchmark.jsonl"
)
DEFAULT_ARTIFACTS_DIR = REPO_ROOT / "artifacts/experimental"

_SOURCE_TYPE_TO_DATATYPE = {
    "string": DataType.STRING,
    "integer": DataType.INTEGER,
    "decimal": DataType.DECIMAL,
    "boolean": DataType.BOOLEAN,
    "timestamp": DataType.TIMESTAMP,
    "date": DataType.DATE,
    "json": DataType.JSON,
    "uuid": DataType.UUID,
}

_SOURCE_TYPE_TO_FIELDTYPE = {
    "string": FieldType.TEXT,
    "integer": FieldType.INTEGER,
    "decimal": FieldType.FLOAT,
    "boolean": FieldType.BOOLEAN,
    "timestamp": FieldType.DATETIME,
    "date": FieldType.DATETIME,
    "json": FieldType.TEXT,
    "uuid": FieldType.CODE,
}

print("Repository root:", REPO_ROOT)
print("Benchmark:", DEFAULT_BENCHMARK)
print("Artifacts directory:", DEFAULT_ARTIFACTS_DIR)


Repository root: d:\memoire\rag\1.0.0
Benchmark: d:\memoire\rag\1.0.0\data\experimental\open_world\benchmark.jsonl
Artifacts directory: d:\memoire\rag\1.0.0\artifacts\experimental


# 2. Mock Retrieval and LLM Boundaries

The original script uses two controlled mock objects:

1. **`BenchmarkRetriever`** simulates `AdvancedRAGRetriever.retrieve_candidates()`.
2. **`BenchmarkLLMOrchestrator`** simulates `LLMOrchestrator.match_field()`.

Their answers come directly from the benchmark file. This is important because the notebook can exercise the complete downstream pipeline without requiring external infrastructure.

Only these I/O boundaries are mocked. The scoring, semantic-decision, MDE-validation, decision-policy, evolution-planning, and migration-building components remain the real project classes.


In [2]:
class BenchmarkRetriever:
    """Mock for AdvancedRAGRetriever.retrieve_candidates.

    Candidate information is read directly from the benchmark.
    """

    def __init__(self, candidates_spec: List[Dict[str, Any]]):
        self._candidates_spec = candidates_spec

    def retrieve_candidates(self, query):
        results = []

        for c in self._candidates_spec:
            table, column = c["target"].split(".", 1)

            doc = KnowledgeBaseDocument(
                id=c["target"],
                table=table,
                column=column,
                content=c["target"],
                metadata={
                    "data_type": c.get("target_data_type", "string"),
                    "constraints": {
                        "is_primary_key": c.get(
                            "is_primary_key", False
                        ),
                        "is_foreign_key": c.get(
                            "is_foreign_key", False
                        ),
                        "not_null": not c.get(
                            "nullable", True
                        ),
                    },
                    "description": c.get(
                        "rationale", ""
                    ),
                },
            )

            results.append(
                (
                    doc,
                    c["embedding_score"],
                    c["embedding_score"],
                )
            )

        return results


class BenchmarkLLMOrchestrator:
    """Mock for LLMOrchestrator.match_field.

    Candidate scores and rationales are taken from the benchmark.
    """

    def __init__(self, candidates_spec: List[Dict[str, Any]]):
        self._candidates_spec = candidates_spec

    def match_field(
        self,
        query,
        docs,
        bi_scores,
        cross_scores,
    ):
        from types import SimpleNamespace

        candidates = [
            SimpleNamespace(
                target=c["target"],
                confidence_llm=c["llm_score"],
                confidence_model=c["embedding_score"],
                rationale=c.get("rationale", ""),
                guardrails=[],
            )
            for c in self._candidates_spec
        ]

        return SimpleNamespace(
            candidates=candidates
        )


# 3. Benchmark Loading and Source Conversion

The benchmark is stored as JSON Lines (`.jsonl`), where every line represents one source element.

The following helper functions convert each benchmark item into the project's domain entities:

- `SourceField` for semantic retrieval/decision processing.
- `USchemaAttribute` for MDE validation and schema-evolution reasoning.

The type mappings preserve the original script's mapping between benchmark source types and the project's `DataType` / `FieldType` enums.


In [3]:
def load_benchmark(path: Path) -> List[Dict[str, Any]]:
    """Load all non-empty JSONL records from the benchmark."""
    items = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                items.append(json.loads(line))

    return items


def source_field_from_item(
    item: Dict[str, Any]
) -> SourceField:
    """Convert one benchmark item into a SourceField."""
    return SourceField(
        path=item["source_name"],
        name_tokens=item["source_name"].split("."),
        inferred_type=_SOURCE_TYPE_TO_FIELDTYPE.get(
            item["source_type"],
            FieldType.TEXT,
        ),
        hints=[item.get("source_context", "")],
    )


def uschema_attribute_from_item(
    item: Dict[str, Any]
) -> USchemaAttribute:
    """Convert one benchmark item into a USchemaAttribute."""
    leaf = item["source_name"].split(".")[-1]

    return USchemaAttribute(
        name=leaf,
        data_type=_SOURCE_TYPE_TO_DATATYPE.get(
            item["source_type"],
            DataType.STRING,
        ),
        required=False,
        is_key=(leaf == "id"),
    )


# 4. Load and Inspect the Controlled Benchmark

Before executing the pipeline, load the benchmark and inspect its records.

This is useful in a notebook because every experiment can be inspected interactively before running the complete pipeline.


In [4]:
benchmark = load_benchmark(DEFAULT_BENCHMARK)

print(f"Loaded {len(benchmark)} benchmark records.")
print()

for item in benchmark:
    print(
        f"{item['source_id']}: "
        f"{item['source_name']} -> "
        f"{item['gold_decision']}"
    )


Loaded 5 benchmark records.

s1: patient.id -> MATCH
s2: patient.name -> MATCH
s3: patient.device.id -> REVIEW
s4: patient.new_score -> EVOLVE
s5: patient.internal_debug -> REJECT


# 5. Initialize the Real Domain Services

These are the actual project services used by the experimental pipeline.

The decision thresholds correspond to the command-line defaults from the original script:

- `tau_match = 0.85`
- `tau_review = 0.70`

The pipeline also uses the project's `RelevancePolicy`, `EvolutionPlanner`, and `MigrationBuilder`.


In [5]:
TOP_K = 10
THRESHOLD_MATCH = 0.85
THRESHOLD_REVIEW = 0.70

scoring_system = HybridScoringSystem()
mde_validator = MDEValidator()
relevance_policy = RelevancePolicy()

decision_policy = DecisionPolicy(
    thresholds=DecisionThresholds(
        tau_match=THRESHOLD_MATCH,
        tau_review=THRESHOLD_REVIEW,
    ),
    relevance_policy=relevance_policy,
)

evolution_planner = EvolutionPlanner()
migration_builder = MigrationBuilder()

print("Real domain services initialized.")


Real domain services initialized.


# 6. Prepare Artifact Containers and Counters

The original script produces several experimental artifacts.

The notebook keeps the same artifact structure:

- `candidates.jsonl`
- `semantic_decisions.jsonl`
- `mde_validations.jsonl`
- `decisions.jsonl`
- `evolution_operations.jsonl`
- `sql_migrations.jsonl`
- `summary.json`

The lists below collect records while the pipeline runs.


In [6]:
candidates_records: List[Dict[str, Any]] = []
semantic_decisions_records: List[Dict[str, Any]] = []
mde_records: List[Dict[str, Any]] = []
decisions_records: List[Dict[str, Any]] = []
evolution_records: List[Dict[str, Any]] = []

all_schema_changes = []

mde_status_counts: Dict[str, int] = {}
decision_counts: Dict[str, int] = {}
change_type_counts: Dict[str, int] = {}


# 7. Execute the Experimental Pipeline

This is the central notebook cell.

For every benchmark source element, the pipeline:

1. Converts the source to project domain entities.
2. Creates the benchmark-backed retriever and LLM mock.
3. Calls the real `SemanticDecisionService`.
4. Records all semantic candidates and scores.
5. Selects the best semantic candidate.
6. Runs the real `MDEValidator`.
7. Applies the real `DecisionPolicy`.
8. If the result is `EVOLVE`, calls the real `EvolutionPlanner`.
9. Collects schema changes.
10. Builds SQL through the real `MigrationBuilder`.

The logic below follows the original uploaded Python file rather than replacing it with a simplified implementation.


In [7]:
for item in benchmark:
    source_id = item["source_id"]
    source_name = item["source_name"]

    source_field = source_field_from_item(item)
    source_attr = uschema_attribute_from_item(item)

    candidates_spec = item.get("candidates", [])

    # Mock only the retrieval and LLM boundaries.
    retriever = BenchmarkRetriever(candidates_spec)
    llm_orchestrator = BenchmarkLLMOrchestrator(
        candidates_spec
    )

    # Use the REAL SemanticDecisionService.
    service = SemanticDecisionService(
        retriever=retriever,
        llm_orchestrator=llm_orchestrator,
        scoring_system=scoring_system,
        top_k=TOP_K,
    )

    semantic_decisions: List[
        SemanticDecision
    ] = service.evaluate_source_element(
        source_field
    )

    # ---------------------------------------------------------------
    # Record candidate and semantic-decision information.
    # ---------------------------------------------------------------
    for sd in semantic_decisions:
        candidates_records.append(
            {
                "source_id": source_id,
                "source_name": source_name,
                "candidate_id": (
                    sd.candidate.target_element
                ),
                "rank": sd.candidate.rank,
                "embedding_score": (
                    sd.embedding_score
                ),
                "llm_score": sd.llm_score,
                "combined_score": (
                    sd.combined_score
                ),
            }
        )

        semantic_decisions_records.append(
            {
                "source_id": source_id,
                "source_name": source_name,
                "candidate": (
                    sd.candidate.target_element
                ),
                "embedding_score": (
                    sd.embedding_score
                ),
                "llm_score": sd.llm_score,
                "combined_score": (
                    sd.combined_score
                ),
                "rationale": sd.rationale,
            }
        )

    # ---------------------------------------------------------------
    # Select the highest combined-score candidate.
    # ---------------------------------------------------------------
    best_semantic_decision: Optional[
        SemanticDecision
    ] = None

    if semantic_decisions:
        best_semantic_decision = max(
            semantic_decisions,
            key=lambda d: d.combined_score,
        )

    # ---------------------------------------------------------------
    # MDE validation of the best candidate.
    # ---------------------------------------------------------------
    mde_result = None

    if best_semantic_decision is not None:
        best_candidate_spec = next(
            c
            for c in candidates_spec
            if c["target"]
            == best_semantic_decision.candidate.target_element
        )

        table, column = (
            best_candidate_spec["target"].split(
                ".",
                1,
            )
        )

        structural_context = StructuralContext(
            is_primary_key=best_candidate_spec.get(
                "is_primary_key",
                False,
            ),
            is_foreign_key=best_candidate_spec.get(
                "is_foreign_key",
                False,
            ),
            nullable=best_candidate_spec.get(
                "nullable",
                True,
            ),
            table_name=table,
            column_name=column,
        )

        mde_result = mde_validator.validate(
            source=source_attr,
            target_type=best_candidate_spec.get(
                "target_data_type",
                "string",
            ),
            structural_context=structural_context,
        )

        mde_records.append(
            {
                "source_id": source_id,
                "mde_status": (
                    mde_result.status.value
                ),
                "type_score": (
                    mde_result.type_score
                ),
                "structure_score": (
                    mde_result.structural_score
                ),
                "violations": (
                    mde_result.violations
                ),
            }
        )

        mde_status_counts[
            mde_result.status.value
        ] = (
            mde_status_counts.get(
                mde_result.status.value,
                0,
            )
            + 1
        )

    # ---------------------------------------------------------------
    # Apply the real DecisionPolicy.
    # ---------------------------------------------------------------
    decision_result = decision_policy.decide(
        source_element=source_name,
        semantic_decision=(
            best_semantic_decision
        ),
        mde_validation=mde_result,
        semantic_evidence_for_relevance=item.get(
            "semantic_evidence",
            0.0,
        ),
    )

    decision_counts[
        decision_result.decision_type.value
    ] = (
        decision_counts.get(
            decision_result.decision_type.value,
            0,
        )
        + 1
    )

    decision_record = {
        "source_id": source_id,
        "decision": (
            decision_result.decision_type.value
        ),
    }

    # ---------------------------------------------------------------
    # Schema evolution.
    # ---------------------------------------------------------------
    if (
        decision_result.decision_type
        == DecisionType.EVOLVE
    ):
        # Benchmark convention:
        # patient.new_score -> relational table "patient".
        parent = (
            source_name.rsplit(".", 1)[0]
            if "." in source_name
            else None
        )

        schema_change = evolution_planner.plan(
            decision=decision_result,
            source_attribute=source_attr,
            target_table=parent,
        )

        if schema_change is not None:
            all_schema_changes.append(
                schema_change
            )

            change_type_counts[
                schema_change.change_type.name
            ] = (
                change_type_counts.get(
                    schema_change.change_type.name,
                    0,
                )
                + 1
            )

            decision_record.update(
                {
                    "change_type": (
                        schema_change.change_type.value
                    ),
                    "target_table": (
                        schema_change.target_table
                    ),
                    "target_column": (
                        schema_change.target_column
                    ),
                }
            )

            evolution_records.append(
                {
                    "source_id": source_id,
                    "change_type": (
                        schema_change.change_type.value
                    ),
                    "target_table": (
                        schema_change.target_table
                    ),
                    "target_column": (
                        schema_change.target_column
                    ),
                    "data_type": (
                        schema_change.data_type
                    ),
                    "predicted_operation": (
                        schema_change.change_type.value
                    ),
                    "gold_operation": item.get(
                        "expected_operation"
                    ),
                    "operation_match": (
                        schema_change.change_type.value
                        == "add_column"
                        and item.get(
                            "expected_operation"
                        )
                        == "ADD_COLUMN"
                    ),
                }
            )

    decisions_records.append(
        decision_record
    )


# 8. Generate SQL Migration Statements

After processing all benchmark elements, the collected schema changes are passed to the real `MigrationBuilder`.

If no schema changes exist, the pipeline generates an empty SQL list.


In [8]:
sql_statements = (
    migration_builder.build_migration(
        all_schema_changes
    )
    if all_schema_changes
    else []
)

print("Generated SQL statements:")
for sql in sql_statements:
    print(" ", sql)


Generated SQL statements:
  ALTER TABLE patient ADD COLUMN new_score DECIMAL(10,2);


# 9. Build the Experimental Summary

The summary reproduces the key measurements generated by the original pipeline.

It reports:

- number of source elements,
- retrieved candidates,
- semantic decisions,
- MDE validation statuses,
- final decisions,
- evolution operations,
- generated SQL statements.


In [9]:
summary = {
    "run_type": "controlled_mock",
    "source_elements": len(benchmark),
    "retrieved_candidates": len(
        candidates_records
    ),
    "semantic_decisions": len(
        semantic_decisions_records
    ),
    "mde_validations": mde_status_counts,
    "final_decisions": decision_counts,
    "evolution_operations": change_type_counts,
    "sql_statements_generated": len(
        sql_statements
    ),
    "sql_statements": sql_statements,
}

summary


{'run_type': 'controlled_mock',
 'source_elements': 5,
 'retrieved_candidates': 3,
 'semantic_decisions': 3,
 'mde_validations': {'VALID': 3},
 'final_decisions': {'MATCH': 2, 'REVIEW': 1, 'EVOLVE': 1, 'REJECT': 1},
 'evolution_operations': {'ADD_COLUMN': 1},
 'sql_statements_generated': 1,
 'sql_statements': ['ALTER TABLE patient ADD COLUMN new_score DECIMAL(10,2);']}

# 10. Persist Experimental Artifacts

This cell writes the same artifact files produced by the original script.

Because a notebook is interactive, the artifacts are still written to the project so that the execution remains reproducible and compatible with the rest of the repository.


In [10]:
def write_jsonl(
    path: Path,
    records: List[Dict[str, Any]],
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        for record in records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )


def write_json(
    path: Path,
    obj: Dict[str, Any],
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
        )


DEFAULT_ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

write_jsonl(
    DEFAULT_ARTIFACTS_DIR / "candidates.jsonl",
    candidates_records,
)

write_jsonl(
    DEFAULT_ARTIFACTS_DIR / "semantic_decisions.jsonl",
    semantic_decisions_records,
)

write_jsonl(
    DEFAULT_ARTIFACTS_DIR / "mde_validations.jsonl",
    mde_records,
)

write_jsonl(
    DEFAULT_ARTIFACTS_DIR / "decisions.jsonl",
    decisions_records,
)

write_jsonl(
    DEFAULT_ARTIFACTS_DIR / "evolution_operations.jsonl",
    evolution_records,
)

write_jsonl(
    DEFAULT_ARTIFACTS_DIR / "sql_migrations.jsonl",
    [{"sql": s} for s in sql_statements],
)

write_json(
    DEFAULT_ARTIFACTS_DIR / "summary.json",
    summary,
)

print(
    f"Artifacts written to: {DEFAULT_ARTIFACTS_DIR}"
)


Artifacts written to: d:\memoire\rag\1.0.0\artifacts\experimental


# 11. Human-Readable Experimental Report

The original command-line script prints a compact summary. This notebook provides the same information in a notebook-friendly form.

This cell is intentionally presentation-oriented: it does not alter the experimental results.


In [11]:
print("=" * 60)
print("EXPERIMENTAL PIPELINE")
print("=" * 60)

print(
    f"Source elements: "
    f"{summary['source_elements']}"
)

print(
    f"Retrieved candidates: "
    f"{summary['retrieved_candidates']}"
)

print(
    f"Semantic decisions: "
    f"{summary['semantic_decisions']}"
)

print()
print("MDE validations:")

for status in (
    "VALID",
    "VALID_WITH_TRANSFORMATION",
    "REVIEW",
    "INVALID",
):
    print(
        f"  {status}: "
        f"{summary['mde_validations'].get(status, 0)}"
    )

print()
print("Final decisions:")

for decision in (
    "MATCH",
    "EVOLVE",
    "REVIEW",
    "REJECT",
):
    print(
        f"  {decision}: "
        f"{summary['final_decisions'].get(decision, 0)}"
    )

print()
print("Evolution operations:")

for operation in (
    "ADD_TABLE",
    "CREATE_TABLE",
    "ADD_COLUMN",
    "ADD_FOREIGN_KEY",
    "ADD_ASSOCIATION_TABLE",
):
    print(
        f"  {operation}: "
        f"{summary['evolution_operations'].get(operation, 0)}"
    )

print()
print(
    f"SQL statements generated: "
    f"{summary['sql_statements_generated']}"
)

print()
print("Execution: SKIPPED (dry-run)")

print("=" * 60)


EXPERIMENTAL PIPELINE
Source elements: 5
Retrieved candidates: 3
Semantic decisions: 3

MDE validations:
  VALID: 3
  VALID_WITH_TRANSFORMATION: 0
  REVIEW: 0
  INVALID: 0

Final decisions:
  MATCH: 2
  EVOLVE: 1
  REVIEW: 1
  REJECT: 1

Evolution operations:
  ADD_TABLE: 0
  CREATE_TABLE: 0
  ADD_COLUMN: 1
  ADD_FOREIGN_KEY: 0
  ADD_ASSOCIATION_TABLE: 0

SQL statements generated: 1

Execution: SKIPPED (dry-run)


# 12. Inspect Candidate-Level Results

A notebook makes it easier to inspect individual intermediate results.

The following cell displays the candidate-level information generated by the semantic-decision stage.


In [12]:
try:
    import pandas as pd

    candidates_df = pd.DataFrame(
        candidates_records
    )

    display(candidates_df)
except ImportError:
    print(
        "pandas is not installed; "
        "showing raw records instead."
    )
    for record in candidates_records:
        print(record)


,source_id,source_name,candidate_id,rank,embedding_score,llm_score,combined_score
0,s1,patient.id,patient.id,1,0.96,0.97,1.0000
1,s2,patient.name,patient.name,1,0.93,0.95,0.9765
2,s3,patient.device.id,patient.device_id,1,0.72,0.68,0.7730


# 13. Inspect Final Decisions

This view focuses on the final decision produced for each source element.

It is particularly useful for checking the expected experimental categories: `MATCH`, `EVOLVE`, `REVIEW`, and `REJECT`.


In [13]:
try:
    decisions_df = pd.DataFrame(
        decisions_records
    )
    display(decisions_df)
except NameError:
    for record in decisions_records:
        print(record)


,source_id,decision,change_type,target_table,target_column
0,s1,MATCH,NaN,NaN,NaN
1,s2,MATCH,NaN,NaN,NaN
2,s3,REVIEW,NaN,NaN,NaN
3,s4,EVOLVE,add_column,patient,new_score
4,s5,REJECT,NaN,NaN,NaN


# 14. Inspect MDE Validation Results

The MDE validation stage evaluates the best semantic candidate structurally and by data type.

The following table exposes the resulting status, type score, structural score, and any reported violations.


In [14]:
try:
    mde_df = pd.DataFrame(
        mde_records
    )
    display(mde_df)
except NameError:
    for record in mde_records:
        print(record)


,source_id,mde_status,type_score,structure_score,violations
0,s1,VALID,1.0,1.0,[]
1,s2,VALID,1.0,1.0,[]
2,s3,VALID,1.0,1.0,[]


# 15. Inspect Schema-Evolution Operations

Only source elements classified as `EVOLVE` reach the evolution planner.

This table shows the operation predicted by the planner and, when available in the benchmark, the expected operation.


In [15]:
try:
    evolution_df = pd.DataFrame(
        evolution_records
    )
    display(evolution_df)
except NameError:
    for record in evolution_records:
        print(record)


,source_id,change_type,target_table,target_column,data_type,predicted_operation,gold_operation,operation_match
0,s4,add_column,patient,new_score,"DECIMAL(10,2)",add_column,ADD_COLUMN,True


# 16. SQL Migration Output

The final SQL statements are generated by the project's `MigrationBuilder`.

The notebook intentionally does not execute them against PostgreSQL. This corresponds to the original dry-run behavior.


In [16]:
if sql_statements:
    for index, sql in enumerate(
        sql_statements,
        start=1,
    ):
        print(
            f"{index}. {sql}"
        )
else:
    print(
        "No SQL migration statements were generated."
    )


1. ALTER TABLE patient ADD COLUMN new_score DECIMAL(10,2);


# 17. Reproducibility Notes

## What this notebook reproduces

This notebook reproduces the experimental controlled-mock execution represented by the uploaded Python script.

### Mocked components

- Retrieval boundary
- LLM boundary

### Real components

- `SemanticDecisionService`
- `HybridScoringSystem`
- `MDEValidator`
- `RelevancePolicy`
- `DecisionPolicy`
- `EvolutionPlanner`
- `MigrationBuilder`

### Important scientific limitation

Because the retriever and LLM responses are supplied by a small hand-built benchmark, this execution should be interpreted as **integration evidence**, not as a scientific evaluation of retrieval quality or LLM performance.

For a scientific experiment, the retrieval layer should instead operate on the real dataset/vector store and the evaluation should report the appropriate retrieval and decision metrics.


# 18. Optional: Reset and Re-run the Experiment

Run this cell if you want to execute the complete notebook experiment again from a clean in-memory state.

Restarting the Jupyter kernel is the strongest reset because it clears all Python objects. The cell below is provided mainly as a reminder of the correct execution order.


In [17]:
# Recommended reproducible execution order:
#
# 1. Restart kernel
# 2. Run all cells from top to bottom
#
# The benchmark is deterministic, so repeated runs with the same
# project code and benchmark should produce the same controlled results.
print("Ready for a clean top-to-bottom notebook execution.")


Ready for a clean top-to-bottom notebook execution.
